# Screencast 3 — Pruning de Modelos
**Objetivo deste notebook:**

Carregar o **modelo de referência** (ResNet50 + transfer learning) treinado no Screencast 1 e aplicar **poda estruturada** (*structured pruning*) com a biblioteca [`torch-pruning`](https://github.com/VainF/Torch-Pruning), que constrói um grafo de dependências do modelo e remove fisicamente canais inteiros das convoluções — inclusive nas conexões residuais da ResNet50 — em vez de apenas zerar pesos com uma máscara.

Este notebook foca exclusivamente em poda **estruturada**: é a técnica que de fato reduz o número de parâmetros e MACs do modelo, e por isso é a única com potencial real de ganho de latência em hardware convencional (CPU/GPU sem suporte a esparsidade). Todas as medições (tamanho, latência, acurácia) rodam em **FP32/CPU** — a mesma configuração `fp32_cpu` usada como linha de base no Screencast 2, para comparação direta entre os dois screencasts.

Variantes exploradas:
1. **Baseline densa em FP32/CPU (Variante 0)** — o modelo do Screencast 1, sem nenhuma poda, medido na mesma configuração usada como linha de base no Screencast 2.
2. **Poda estruturada *zero-shot* (30%)** — `torch-pruning` remove fisicamente 30% dos canais de cada convolução, por magnitude (norma L2), de uma só vez e **sem fine-tuning** — para medir o custo bruto de "podar e servir".
3. **Poda estruturada + fine-tuning (10%, 20%, 30%)** — a mesma poda física via `torch-pruning`, em três intensidades, cada uma seguida de fine-tuning para recuperar a acurácia perdida — para medir o trade-off entre redução de parâmetros/latência e queda de acurácia em cada ponto.
4. **Comparação final** — parâmetros removidos, tamanho em disco (bruto e comprimido), latência e acurácia de todas as variantes, incluindo os resultados de quantização do Screencast 2 para contextualizar qual técnica comprime mais.


## 1. Ambiente e Carregamento do Modelo de Referência

Este notebook assume que o **Screencast 1**, já foi executado e que os seguintes arquivos existem:

- `models/edge_ai_reference_resnet50_pytorch.pt`
- `models/baseline_metrics_pytorch.json`
- O dataset extraído em `data/` (mesma estrutura `train/`, `test/`).

Se o **Screencast 2 (`02_quantizacao.ipynb`)** também já tiver sido executado, carregamos aqui `models/quantization_comparison_pytorch.json` apenas como referência para a comparação final (Seção 7) — não é necessário para o pruning em si.

Usamos a biblioteca [`torch-pruning`](https://github.com/VainF/Torch-Pruning) para toda a poda deste notebook: diferente de `torch.nn.utils.prune` (que apenas aplica uma máscara sobre os pesos, sem mudar o *shape* dos tensores), `torch-pruning` constrói um grafo de dependências do modelo e remove fisicamente os canais podados, coordenando a remoção entre camadas acopladas — essencial em uma ResNet, onde a saída de um bloco é somada ao atalho (*shortcut*) e ambos os lados da soma precisam remover exatamente os mesmos índices de canal.

Como a poda é física (os tensores ficam menores, não apenas zerados), o fine-tuning entre as variantes pode rodar em GPU (mais rápido), se disponível. Mas para que a comparação de tamanho, latência e acurácia seja diretamente comparável ao Screencast 2, **todas as medições de inferência (tamanho em disco, latência, acurácia) deste notebook rodam em CPU** — a mesma configuração `fp32_cpu` usada lá como linha de base. Ou seja: `DEVICE` (GPU, se disponível) é usado só para treinar/fazer fine-tuning; `DEVICE_EVAL` (sempre CPU) é usado para medir e salvar cada variante.


In [ ]:
import os
import time
import json
import copy
import gzip
import shutil
import pathlib

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

# torch-pruning constrói um grafo de dependências do modelo e remove fisicamente os canais
# podados (diferente de torch.nn.utils.prune, que só aplica uma máscara) — é a única biblioteca
# de poda usada neste notebook, já que aqui exploramos apenas poda estruturada.
import torch_pruning as tp

print("PyTorch:", torch.__version__)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo de treino/fine-tuning:", DEVICE)

# Todas as medições de inferência (tamanho, latência, acurácia) deste notebook rodam em CPU —
# a mesma configuração "fp32_cpu" usada como linha de base no Screencast 2 — para que a
# comparação entre pruning e quantização seja direta, sem misturar o efeito do hardware.
DEVICE_EVAL = torch.device("cpu")
print("Dispositivo de avaliação (fixo, para comparabilidade com o Screencast 2):", DEVICE_EVAL)

MODELS_DIR = pathlib.Path("../models")
DATA_DIR = pathlib.Path("../data")

REFERENCE_MODEL_PATH = MODELS_DIR / "edge_ai_reference_resnet50_pytorch.pt"
BASELINE_METRICS_PATH = MODELS_DIR / "baseline_metrics_pytorch.json"
QUANT_COMPARISON_PATH = MODELS_DIR / "quantization_comparison_pytorch.json"

assert REFERENCE_MODEL_PATH.exists(), (
    "Modelo de referencia (PyTorch) nao encontrado. Execute o Screencast 1, versao PyTorch "
    "(01_setup_pytorch.ipynb), primeiro."
)

with open(BASELINE_METRICS_PATH, "r", encoding="utf-8") as f:
    baseline_metrics = json.load(f)

print(json.dumps(baseline_metrics, indent=2, ensure_ascii=False))

quantization_comparison = None
if QUANT_COMPARISON_PATH.exists():
    with open(QUANT_COMPARISON_PATH, "r", encoding="utf-8") as f:
        quantization_comparison = json.load(f)
    print("\nComparacao de quantizacao (Screencast 2) carregada - sera usada na Secao 7.")
else:
    print("\nAviso: comparacao de quantizacao (Screencast 2) nao encontrada - a Secao 7 seguira sem ela.")


In [ ]:
# weights_only=False: o Screencast 1 salvou o objeto do modelo inteiro (torch.save(model, path)),
# não apenas os pesos (state_dict) — precisamos desserializar a arquitetura completa também.
reference_model = torch.load(REFERENCE_MODEL_PATH, map_location="cpu", weights_only=False)
reference_model.eval()
# Fica em CPU por padrão — cada variante abaixo decide para qual dispositivo copiar
# (DEVICE para fine-tuning, DEVICE_EVAL para medição/salvamento).

IMG_SIZE = tuple(baseline_metrics["img_size"])
print(f"Modelo de referencia carregado. Entrada esperada: {(3,) + IMG_SIZE}")


## 2. Recarregando o Pipeline de Dados

Mesma configuração dos Screencasts anteriores — precisamos do conjunto de teste (avaliação), do split de validação (early stopping do fine-tuning pós-poda) e do conjunto de treino (fine-tuning propriamente dito).


In [ ]:
def find_split_dirs(root: pathlib.Path):
    train_dirs = list(root.rglob("train"))
    test_dirs = list(root.rglob("test"))
    train_dir = next((d for d in train_dirs if (d / "ok_front").exists()), None)
    test_dir = next((d for d in test_dirs if (d / "ok_front").exists()), None)
    return train_dir, test_dir

TRAIN_DIR, TEST_DIR = find_split_dirs(DATA_DIR)
assert TRAIN_DIR is not None and TEST_DIR is not None, "Dataset não encontrado em ./data — execute o Screencast 1 primeiro."

BATCH_SIZE = 32
SEED = 42
torch.manual_seed(SEED)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

data_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

CLASS_TO_IDX = {"ok_front": 0, "def_front": 1}

def remap_target(default_class_to_idx):
    inverse_default = {v: k for k, v in default_class_to_idx.items()}
    return lambda idx: CLASS_TO_IDX[inverse_default[idx]]

full_train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=data_transform)
full_train_dataset.target_transform = remap_target(full_train_dataset.class_to_idx)

test_dataset = datasets.ImageFolder(TEST_DIR, transform=data_transform)
test_dataset.target_transform = remap_target(test_dataset.class_to_idx)

n_val = int(len(full_train_dataset) * 0.2)
n_train = len(full_train_dataset) - n_val
train_dataset, val_dataset = random_split(
    full_train_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
# shuffle=True (com seed fixa) — mesma razão do Screencast 2: sem shuffle, ImageFolder agrupa
# os arquivos por classe, e qualquer corte por MAX_EVAL_SAMPLES enviesaria a acurácia medida.
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    generator=torch.Generator().manual_seed(SEED),
)

print(f"Treino: {n_train} imagens | Validação: {n_val} imagens | Teste: {len(test_dataset)} imagens")


## 3. Funções Utilitárias — Parâmetros, Tamanho, Latência e Acurácia

Diferente da quantização (Screencast 2), a poda estruturada não muda o **dtype** dos tensores — continuam `float32` — mas, como `torch-pruning` remove canais fisicamente, o *shape* de cada tensor podado fica menor. Ou seja, ao contrário de uma poda por máscara, **o `.pt` bruto de um modelo podado com `torch-pruning` já fica menor**, sem precisar de nenhum passo extra de compressão para isso.

Ainda assim, medimos tamanho bruto **e** comprimido (`gzip`) para manter a comparação direta com o Screencast 2. A métrica central desta vez é a **redução de parâmetros** (`param_reduction`): a fração de parâmetros do modelo original que deixou de existir após a poda — não há mais "zeros para contar", os pesos correspondentes foram removidos.


In [ ]:
def count_params(model, example_inputs):
    _, params = tp.utils.count_ops_and_params(model, example_inputs)
    return params


def get_ignored_layers(model):
    # A cabeça de classificação (Linear) fica de fora: mudar o número de saídas mudaria o contrato do modelo.
    return [m for m in model.modules() if isinstance(m, nn.Linear)]


def save_model(model, path: pathlib.Path):
    # Mesma lógica de serialização do Screencast 1/2: objeto do modelo inteiro (arquitetura + pesos).
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model, path)
    raw_mb = path.stat().st_size / (1024 * 1024)

    # Compressão gzip do arquivo salvo — ainda vale medir, mesmo com poda física: os pesos
    # remanescentes têm a mesma redundância que qualquer tensor float32 treinado.
    gz_path = path.with_suffix(path.suffix + ".gz")
    with open(path, "rb") as f_in, gzip.open(gz_path, "wb", compresslevel=9) as f_out:
        shutil.copyfileobj(f_in, f_out)
    gz_mb = gz_path.stat().st_size / (1024 * 1024)

    return raw_mb, gz_mb


def measure_latency_ms(model, sample_batch, device=DEVICE_EVAL, n_runs=100):
    model.eval()
    sample_batch = sample_batch.to(device)
    with torch.no_grad():
        _ = model(sample_batch)  # warm-up
        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            _ = model(sample_batch)
            times.append((time.perf_counter() - start) * 1000)
    return float(np.mean(times)), float(np.std(times))


def evaluate_accuracy(model, loader, device=DEVICE_EVAL, max_samples=None):
    model.eval()
    y_true, y_pred = [], []
    seen = 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.sigmoid(logits).cpu().numpy().ravel()
            preds = (probs >= 0.5).astype(int)
            y_pred.extend(preds.tolist())
            y_true.extend(labels.numpy().ravel().tolist())
            seen += len(preds)
            if max_samples is not None and seen >= max_samples:
                break

    if max_samples is not None:
        y_true, y_pred = y_true[:max_samples], y_pred[:max_samples]

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return float((y_true == y_pred).mean())


In [ ]:
# Amostra fixa (1 imagem) para medição de latência — mesma amostra usada em todas as variantes
sample_batch, _ = next(iter(test_loader))
LATENCY_SAMPLE = sample_batch[:1]

# Amostra de avaliação — use um valor menor (ex.: 200) para iterar mais rápido durante a gravação;
# use None para avaliar no conjunto de teste completo antes da entrega final.
MAX_EVAL_SAMPLES = None

results = {}  # nome_da_variante -> {"param_reduction", "size_mb", "size_mb_gzip", "latency_ms_mean", "latency_ms_std", "accuracy"}

MODELS_DIR_PRUNE = MODELS_DIR / "pytorch_prune"
MODELS_DIR_PRUNE.mkdir(parents=True, exist_ok=True)


## 4. Variante 0 — Baseline Densa em FP32/CPU (a mesma do Screencast 2)

Esta é exatamente a configuração `fp32_cpu` do Screencast 2: o modelo do Screencast 1, em ponto flutuante (FP32), medido em CPU (`DEVICE_EVAL`). É o ponto de comparação direto — em parâmetros, tamanho (bruto e comprimido) e latência — para as variantes de poda estruturada a seguir, e comparável número a número com a linha de base de quantização.


In [ ]:
dense_model = copy.deepcopy(reference_model).to(DEVICE_EVAL)
dense_model.eval()

raw_mb, gz_mb = save_model(dense_model, MODELS_DIR_PRUNE / "model_dense_baseline.pt")
latency_mean, latency_std = measure_latency_ms(dense_model, LATENCY_SAMPLE, device=DEVICE_EVAL)
accuracy = evaluate_accuracy(dense_model, test_loader, device=DEVICE_EVAL, max_samples=MAX_EVAL_SAMPLES)

results["dense_baseline"] = {
    "param_reduction": 0.0, "size_mb": raw_mb, "size_mb_gzip": gz_mb,
    "latency_ms_mean": latency_mean, "latency_ms_std": latency_std, "accuracy": accuracy,
}
print(results["dense_baseline"])


## 5. Variante 1 — Poda Estruturada *Zero-Shot* com `torch-pruning` (30%, sem Fine-Tuning)

`torch-pruning` constrói um grafo de dependências do modelo e remove fisicamente 30% dos canais de saída de cada convolução, escolhidos globalmente por magnitude (norma L2) — inclusive coordenando a remoção nas conexões residuais da ResNet50 (o último `Conv2d` de cada bloco *bottleneck* e a convolução de `downsample` precisam remover exatamente os mesmos índices de canal, senão a soma do atalho quebra de *shape*). A cabeça de classificação (`Linear`) fica de fora (`ignored_layers`): mudar o número de saídas mudaria o contrato do modelo.

Aplicamos a poda **de uma só vez, sem nenhum fine-tuning** — o objetivo é medir o custo bruto de "podar e servir" uma poda estruturada, como ponto de comparação para a Variante 2 (mesma técnica, mas com fine-tuning). Como não há treino nesta variante, trabalhamos direto em `DEVICE_EVAL` (CPU).


In [ ]:
STRUCTURED_PRUNE_RATIO_ZEROSHOT = 0.3  # 30% dos canais de cada Conv2d, por norma L2, sem fine-tuning

# Sem fine-tuning nesta variante — não há treino, então já trabalhamos direto em DEVICE_EVAL (CPU).
zeroshot_model = copy.deepcopy(reference_model).to(DEVICE_EVAL)
example_inputs = LATENCY_SAMPLE.to(DEVICE_EVAL)
ignored_layers = get_ignored_layers(zeroshot_model)

base_params = count_params(zeroshot_model, example_inputs)

importance = tp.importance.MagnitudeImportance(p=2)
pruner = tp.pruner.MagnitudePruner(
    zeroshot_model,
    example_inputs,
    importance=importance,
    pruning_ratio=STRUCTURED_PRUNE_RATIO_ZEROSHOT,
    ignored_layers=ignored_layers,
)
pruner.step()  # remove fisicamente os canais podados -> tensores ficam menores, não apenas zerados

pruned_params = count_params(zeroshot_model, example_inputs)
param_reduction = 1 - pruned_params / base_params
print(f"Parâmetros: {base_params/1e6:.3f}M -> {pruned_params/1e6:.3f}M ({param_reduction:.1%} de redução)")

zeroshot_model.eval()
raw_mb, gz_mb = save_model(zeroshot_model, MODELS_DIR_PRUNE / "model_structured_zeroshot_30.pt")
latency_mean, latency_std = measure_latency_ms(zeroshot_model, LATENCY_SAMPLE, device=DEVICE_EVAL)
accuracy = evaluate_accuracy(zeroshot_model, test_loader, device=DEVICE_EVAL, max_samples=MAX_EVAL_SAMPLES)

results["structured_zeroshot_30"] = {
    "param_reduction": param_reduction, "size_mb": raw_mb, "size_mb_gzip": gz_mb,
    "latency_ms_mean": latency_mean, "latency_ms_std": latency_std, "accuracy": accuracy,
}
print(results["structured_zeroshot_30"])


In [ ]:
queda_acc_zeroshot = results["dense_baseline"]["accuracy"] - results["structured_zeroshot_30"]["accuracy"]
print(f"Queda de acurácia (densa -> poda estruturada zero-shot, 30%): {queda_acc_zeroshot:.4f}")

if queda_acc_zeroshot > 0.02:
    print(
        "\nQueda de acurácia relevante (> 2 pontos percentuais). "
        "Este é exatamente o cenário em que fine-tuning pós-poda se justifica — "
        "vamos explorar isso na próxima seção, em três intensidades de poda (10%, 20%, 30%)."
    )
else:
    print(
        "\nQueda de acurácia pequena mesmo sem fine-tuning. Ainda assim, vamos treinar as versões "
        "com fine-tuning a seguir para comparar o trade-off em cada intensidade de poda."
    )


## 6. Variante 2 — Poda Estruturada + Fine-Tuning com `torch-pruning` (10%, 20%, 30%)

Repetimos a mesma poda estruturada da Variante 1 — grafo de dependências via `torch-pruning`, remoção física por magnitude (norma L2), cabeça de classificação preservada — mas agora em três intensidades (10%, 20%, 30% dos canais), cada uma partindo de uma cópia independente do modelo de referência e seguida de fine-tuning para recuperar a acurácia perdida.

Rodar as três intensidades lado a lado permite visualizar, na comparação final, o trade-off entre quanto se reduz (parâmetros/latência) e quanto se recupera de acurácia com fine-tuning — e comparar diretamente com a Variante 1 (mesmo alvo de 30%, mas sem fine-tuning).


In [ ]:
def run_epoch(model, loader, optimizer, criterion, training, device=DEVICE):
    model.train(mode=training)
    total_loss, total_correct, total_samples = 0.0, 0, 0

    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            if training:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
            total_loss += loss.item() * labels.size(0)

    return total_loss / total_samples, total_correct / total_samples


def fit(model, optimizer, train_loader, val_loader, epochs, patience=1, device=DEVICE):
    criterion = nn.BCEWithLogitsLoss()
    history = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(model, train_loader, optimizer, criterion, training=True, device=device)
        val_loss, val_acc = run_epoch(model, val_loader, optimizer, criterion, training=False, device=device)

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(
            f"Epoch {epoch}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} "
            f"- val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping na época {epoch} (sem melhora em {patience} épocas).")
                break

    if best_state is not None:
        model.load_state_dict(best_state)  # restore_best_weights=True

    return history


In [ ]:
FINETUNE_PRUNE_RATIOS = [0.10, 0.20, 0.30]  # intensidades de poda estruturada avaliadas com fine-tuning
EPOCHS_FINETUNE = 2

for ratio in FINETUNE_PRUNE_RATIOS:
    pct = round(ratio * 100)
    print(f"\n=== Poda estruturada + fine-tuning: {pct}% ===")

    ft_model = copy.deepcopy(reference_model).to(DEVICE)  # fine-tuning roda em DEVICE (GPU, se disponível)
    example_inputs = LATENCY_SAMPLE.to(DEVICE)
    ignored_layers = get_ignored_layers(ft_model)

    base_params = count_params(ft_model, example_inputs)

    importance = tp.importance.MagnitudeImportance(p=2)
    pruner = tp.pruner.MagnitudePruner(
        ft_model,
        example_inputs,
        importance=importance,
        pruning_ratio=ratio,
        ignored_layers=ignored_layers,
    )
    pruner.step()

    pruned_params = count_params(ft_model, example_inputs)
    param_reduction = 1 - pruned_params / base_params
    print(f"Parâmetros: {base_params/1e6:.3f}M -> {pruned_params/1e6:.3f}M ({param_reduction:.1%} de redução)")

    optimizer = torch.optim.Adam(ft_model.parameters(), lr=1e-5)  # LR pequeno: fine-tuning, não retreino do zero
    fit(ft_model, optimizer, train_loader, val_loader, epochs=EPOCHS_FINETUNE, patience=1, device=DEVICE)

    # Medição e salvamento sempre em CPU (DEVICE_EVAL), para comparabilidade com as demais variantes.
    ft_model.to(DEVICE_EVAL)
    ft_model.eval()
    raw_mb, gz_mb = save_model(ft_model, MODELS_DIR_PRUNE / f"model_structured_finetuned_{pct}.pt")
    latency_mean, latency_std = measure_latency_ms(ft_model, LATENCY_SAMPLE, device=DEVICE_EVAL)
    accuracy = evaluate_accuracy(ft_model, test_loader, device=DEVICE_EVAL, max_samples=MAX_EVAL_SAMPLES)

    results[f"structured_finetuned_{pct}"] = {
        "param_reduction": param_reduction, "size_mb": raw_mb, "size_mb_gzip": gz_mb,
        "latency_ms_mean": latency_mean, "latency_ms_std": latency_std, "accuracy": accuracy,
    }
    print(results[f"structured_finetuned_{pct}"])

    del ft_model  # libera memória antes da próxima intensidade


## 7. Comparação Final

Reunimos aqui a baseline densa (Variante 0), a poda estruturada *zero-shot* (Variante 1) e as três intensidades de poda estruturada com fine-tuning (Variante 2). Quando disponível, incluímos também a melhor variante de quantização do Screencast 2 (`int8_static_ptq`) apenas como referência de contexto — as duas técnicas atacam dimensões diferentes do problema (remoção de parâmetros vs. precisão numérica), e comparar os dois screencasts lado a lado ajuda a decidir a estratégia do Screencast 4.


In [ ]:
comparison = dict(results)

if quantization_comparison is not None and "int8_static_ptq" in quantization_comparison:
    ref = quantization_comparison["int8_static_ptq"]
    comparison["reference_int8_static_ptq_screencast2"] = {
        "param_reduction": 0.0,
        "size_mb": ref["size_mb"],
        "size_mb_gzip": None,
        "latency_ms_mean": ref["latency_ms_mean"],
        "latency_ms_std": ref["latency_ms_std"],
        "accuracy": ref["accuracy"],
    }

print(f"{'Variante':<38}{'Reducao params':>15}{'Tam. bruto (MB)':>17}{'Tam. gzip (MB)':>16}{'Latência (ms)':>15}{'Acurácia':>11}")
for name, m in comparison.items():
    gzip_str = f"{m['size_mb_gzip']:.2f}" if m["size_mb_gzip"] is not None else "n/a"
    print(
        f"{name:<38}{m['param_reduction']:>15.2%}{m['size_mb']:>17.2f}{gzip_str:>16}"
        f"{m['latency_ms_mean']:>15.2f}{m['accuracy']:>11.4f}"
    )


In [ ]:
names = list(comparison.keys())
param_reductions = [comparison[n]["param_reduction"] for n in names]
sizes_raw = [comparison[n]["size_mb"] for n in names]
sizes_gzip = [comparison[n]["size_mb_gzip"] if comparison[n]["size_mb_gzip"] is not None else 0 for n in names]
latencies = [comparison[n]["latency_ms_mean"] for n in names]
accuracies = [comparison[n]["accuracy"] for n in names]

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

axes[0].bar(names, param_reductions, color="#8172B2")
axes[0].set_title("Redução de parâmetros")
axes[0].tick_params(axis="x", rotation=30)

width = 0.4
x = np.arange(len(names))
axes[1].bar(x - width / 2, sizes_raw, width, label="bruto", color="#4C72B0")
axes[1].bar(x + width / 2, sizes_gzip, width, label="gzip", color="#64B5CD")
axes[1].set_title("Tamanho em disco (MB)")
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=30, ha="right")
axes[1].legend()

axes[2].bar(names, latencies, color="#DD8452")
axes[2].set_title("Latência média por inferência (ms)")
axes[2].tick_params(axis="x", rotation=30)

axes[3].bar(names, accuracies, color="#55A868")
axes[3].set_title("Acurácia no conjunto de teste")
axes[3].set_ylim(0, 1)
axes[3].tick_params(axis="x", rotation=30)

for ax in axes:
    ax.tick_params(axis="x", labelsize=8)

plt.tight_layout()
plt.show()


In [ ]:
COMPARISON_PATH = MODELS_DIR / "pruning_comparison_pytorch.json"
with open(COMPARISON_PATH, "w", encoding="utf-8") as f:
    json.dump(comparison, f, indent=2, ensure_ascii=False)

print(f"Comparação salva em: {COMPARISON_PATH.resolve()}")


## 8. Checklist e Próximos Passos

- [x] Modelo de referência do Screencast 1 carregado.
- [x] Baseline densa remedida em FP32/CPU — a mesma configuração `fp32_cpu` usada como linha de base no Screencast 2.
- [x] Poda estruturada *zero-shot* (30%) com `torch-pruning` — remoção física de canais, sem fine-tuning.
- [x] Poda estruturada + fine-tuning com `torch-pruning` em três intensidades (10%, 20%, 30%).
- [x] Comparação de redução de parâmetros, tamanho (bruto e gzip), latência e acurácia salva em `models/pruning_comparison_pytorch.json`.
- [x] Modelos podados salvos em `models/pytorch_prune/`.